# FSDP Advanced: Sharding Strategies and Memory Optimization

## Overview

Advanced FSDP techniques for maximum memory efficiency and training throughput.

### Topics Covered
- Sharding strategy selection
- Mixed precision with FSDP
- CPU offloading
- Activation checkpointing

## 1. Sharding Strategies

| Strategy | Memory | Communication | Use Case |
|----------|--------|---------------|----------|
| NO_SHARD | Highest | Lowest | Small models |
| SHARD_GRAD_OP | Medium | Medium | Medium models |
| FULL_SHARD | Lowest | Highest | Large models |
| HYBRID_SHARD | Balanced | Balanced | Multi-node |

In [ ]:
import torch
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp import ShardingStrategy, MixedPrecision, CPUOffload

def get_fsdp_config(model_size_b, gpu_memory_gb=80):
    """Select optimal FSDP configuration based on model size."""
    
    # Memory required (rough estimate)
    mem_required = model_size_b * 18  # 18 bytes per param (mixed precision)
    
    if mem_required < gpu_memory_gb * 0.7:
        strategy = ShardingStrategy.NO_SHARD
        offload = None
    elif mem_required < gpu_memory_gb * 2:
        strategy = ShardingStrategy.SHARD_GRAD_OP
        offload = None
    elif mem_required < gpu_memory_gb * 8:
        strategy = ShardingStrategy.FULL_SHARD
        offload = None
    else:
        strategy = ShardingStrategy.FULL_SHARD
        offload = CPUOffload(offload_params=True)
    
    print(f"Model: {model_size_b}B params")
    print(f"Strategy: {strategy}")
    print(f"CPU Offload: {offload is not None}")
    
    return strategy, offload

get_fsdp_config(7)   # 7B model
get_fsdp_config(70)  # 70B model

## 2. Mixed Precision Configuration

In [ ]:
# FSDP Mixed Precision Policy
bf16_policy = MixedPrecision(
    param_dtype=torch.bfloat16,      # Parameters in BF16
    reduce_dtype=torch.bfloat16,     # Gradient reduction in BF16
    buffer_dtype=torch.bfloat16,     # Buffers in BF16
)

fp16_policy = MixedPrecision(
    param_dtype=torch.float16,
    reduce_dtype=torch.float16,
    buffer_dtype=torch.float16,
)

# Recommended: BF16 for stability
print("Use BF16 on Ampere+ GPUs for best stability")

## 3. Activation Checkpointing

Trade compute for memory by recomputing activations during backward pass.

In [ ]:
from torch.distributed.algorithms._checkpoint.checkpoint_wrapper import (
    checkpoint_wrapper,
    CheckpointImpl,
)

def apply_activation_checkpointing(model, check_fn=None):
    """Apply activation checkpointing to transformer layers."""
    from torch.distributed.fsdp.wrap import transformer_auto_wrap_policy
    import functools
    
    # Default: checkpoint every transformer block
    if check_fn is None:
        check_fn = lambda m: hasattr(m, 'self_attn')  # Transformer block
    
    for name, module in model.named_modules():
        if check_fn(module):
            # Wrap with checkpointing
            wrapped = checkpoint_wrapper(
                module,
                checkpoint_impl=CheckpointImpl.NO_REENTRANT,
            )
            # Replace in parent
            parent_name = '.'.join(name.split('.')[:-1])
            child_name = name.split('.')[-1]
            if parent_name:
                parent = model.get_submodule(parent_name)
                setattr(parent, child_name, wrapped)
    
    return model

## 4. Summary

### Memory Optimization Checklist

1. **Sharding**: FULL_SHARD for large models
2. **Mixed Precision**: BF16 preferred
3. **Activation Checkpointing**: Enable for memory-bound training
4. **CPU Offload**: Last resort for extreme cases